In [ ]:
import os
import dotenv
import wandb
import tensorflow as tf
# import utils_data_io  
# import utils_post_processing 
import utils


In [2]:
# --- Dynamic GPU/CPU Selection ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPUs detected: {[gpu.name for gpu in gpus]}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"Could not set memory growth for GPUs: {e}")
else:
    print("No GPUs detected. Using CPU only.")
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# --- W&B Setup ---
dotenv.load_dotenv()
key = os.environ.get("WANDB_API_KEY")

if key:
    print("Logging into Weights & Biases...")
    wandb.login(key=key)
else:
    print("WARNING: WANDB_API_KEY not found in .env. Logging in anonymously.")
    wandb.login()

wandb.init(project="animal-faces-classification")

# --- Training Configuration ---
NUM_EPOCHS = 20
LEARNING_RATE = 0.00001

GPUs detected: ['/physical_device:GPU:0']


wandb: Currently logged in as: pshashid (pshashid-university-of-maryland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
if __name__ == "__main__":
    try:
        print("--- Calling Data Preparation Pipeline from utils.py ---")
        train_gen, val_gen = utils.main_prep()

        print("\n--- Training Single ResNet50 Model ---")
        single_resnet = utils.train_model(train_gen, val_gen, architecture="ResNet50", epochs=NUM_EPOCHS, lr=LEARNING_RATE)
        print("\n--- Training Homogeneous Ensemble (ResNet50 x3) ---")
        models_homo, preds_homo, acc_homo = utils.homogeneous_ensemble(train_gen, val_gen, num_models=3, epochs=NUM_EPOCHS, lr=LEARNING_RATE)

        print("\n--- Training Heterogeneous Ensemble (ResNet50 + EfficientNetB0 + MobileNetV2) ---")
        architectures = ["ResNet50", "EfficientNetB0", "MobileNetV2"]
        models_hetero, preds_hetero, acc_hetero = utils.heterogeneous_ensemble(train_gen, val_gen, architectures, epochs=NUM_EPOCHS, lr=LEARNING_RATE)
    except Exception as e:
        print(f"\nAn error occurred during pipeline execution: {e}")
    finally:
        if wandb.run:
            wandb.finish()

--- Calling Data Preparation Pipeline from dataset_operations.py ---
--- Starting Data Preparation Pipeline ---
Using Colab cache for faster access to the 'animal-faces' dataset.
Path to dataset files: /kaggle/input/animal-faces


Train data collected. Samples: 14630


,image_path,label
0,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
1,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
2,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
3,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
4,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat



Validation data collected. Samples: 1500


,image_path,label
0,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat
1,/kaggle/input/animal-faces/afhq/val/cat/flickr...,cat
2,/kaggle/input/animal-faces/afhq/val/cat/flickr...,cat
3,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat
4,/kaggle/input/animal-faces/afhq/val/cat/pixaba...,cat



Setting up ImageDataGenerators (including Data Augmentation)...
Found 14630 validated image filenames belonging to 3 classes.
Found 1500 validated image filenames belonging to 3 classes.
Data Augmentation applied. Generators created successfully.

--- Verification of Data Generators ---

[A] Location of Source Images (first 2 paths from train_df):
  - /kaggle/input/animal-faces/afhq/train/cat/pixabay_cat_000354.jpg
  - /kaggle/input/animal-faces/afhq/train/cat/pixabay_cat_002763.jpg
Total Source Images on Disk: 16130

[B] Generated Samples (Source Count) per Generator:
  - Training Samples (train_generator.samples): 14630
  - Validation Samples (val_generator.samples): 1500
  - Batch Size: 32

[B] Batches per Epoch (The number of times augmentation runs per epoch):
  - Training Steps per Epoch: 458
  - Validation Steps per Epoch: 47
--- Verification Complete ---

 Data Preparation Complete. Generators are ready for Transfer Learning.

--- Training Single ResNet50 Model ---

--- Traini

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 123s 227ms/step - accuracy: 0.7386 - loss: 0.6127 - val_accuracy: 0.7007 - val_loss: 0.6662
Epoch 2/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 87s 189ms/step - accuracy: 0.8918 - loss: 0.2848 - val_accuracy: 0.9080 - val_loss: 0.2490
Epoch 3/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9080 - loss: 0.2439 - val_accuracy: 0.9020 - val_loss: 0.2399
Epoch 4/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9105 - loss: 0.2383 - val_accuracy: 0.9027 - val_loss: 0.2521
Epoch 5/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 188ms/step - accuracy: 0.9208 - loss: 0.2057 - val_accuracy: 0.9380 - val_loss: 0.1941
Epoch 6/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 87s 190ms/step - accuracy: 0.9275 - loss: 0.1965 - val_accuracy: 0.9287 - val_loss: 0.1814
Epoch 7/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 86s 187ms/step - accuracy: 0.9312 - loss: 0.1851 - val_accuracy: 0.9387 - val_loss: 0.1582
Epoch 8/20
458/458 ━━━━━━━━━━━━━━━━━━━━ 87s 189ms/step - accuracy: 0.9348 - loss: 

EfficientNetB0_val_accuracy,▁
MobileNetV2_val_accuracy,▁
ResNet50_val_accuracy,▇█▅▅▁
ensemble_val_accuracy,█▁
epoch/accuracy,▇▇▇▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▁▁▃▃▃▄██████
epoch/epoch,▁▁▂▂▂▃▅▅▆▆▁▁▃▄▄▅▇▁▂▂█▁▂▃▅▇▁▂▃▃▄▅█▁▃▆▇▃▅█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,▅▃▃▃▃▂▂▄▃▃▂▂▂▂▂▂▂▂▂▂▃▂▂▃▃▂▂▂▂▂███▇▇▁▁▁▁▁
epoch/val_accuracy,▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆▇▇▇▇▆▇▇▇▇▇▇▇▆▇▁▂▁▂▂██████
epoch/val_loss,▂▂▂▁▁▁▂▂▁▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▃▂▅▇████▁▁▁▁▁
+1,...
